# 2.2.3 日常运动量随机森林预测模型开发与测试

## 任务说明（摘自考试预览）

随着人们健康意识的增强，越来越多的人开始关注日常运动和健康管理。使用提供的训练数据，补全2.2.3.ipynb代码。选择合适的特征，开发一个预测模型，基于个体性别，个体对运动的看法和个人健康评价来预测个体年龄。利用测试工具对模型进行测试，并对测试结果进行分析，完成测试报告，并运用工具对错误原因进行纠正。

详细说明如下：

（1）正确加载数据集，并显示前五行的数据

（2）使用随机森林模型进行模型训练，要求设定自变量和因变量，并根据自变量特征进行模型训练，最终将训练好的模型以文件名2.2.3_model.pkl保存到考生文件夹，结果文件以2.2.3_results.txt保存到考生文件夹。

（3）使用测试工具对模型进行测试，并记录测试结果，命名2.2.3_report.txt，保存到考生文件夹

（4）对测试结果进行详细分析，并编写测试报告，包括模型性能评估、错误分析及改进建议，将答案写到答题卷文件中，答题卷文件命名为“2.2.3.docx”，保存到考生文件夹。

（5）运用工具分析算法中错误案例产生的原因并进行纠正，重新得到模型训练结果，以文件名2.2.3_results_xgb.txt保存到考生文件夹。

（6）将以上代码以及运行结果，以html格式保存并命名为2.2.3.html，保存到考生文件夹，考生文件夹命名为“准考证号+身份证后6位”。

---

以下为**刷题参考模板**：含完整可运行代码（编程题）或答题示例与生成 docx 草稿代码（主观题）。

## 数据：`data/fitness analysis.csv`

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
import pickle
import xgboost as xgb

from pathlib import Path

_csv = Path('data/fitness analysis.csv')
if not _csv.exists():
    root = Path('.').resolve().parent
    alt = next(root.glob('2.2.3_*/data/fitness analysis.csv'), None)
    if alt and alt.is_file():
        _csv = alt
df = pd.read_csv(_csv)
print('数据文件:', _csv)
print(df.head())

df = df.apply(lambda col: col.map(lambda x: x.strip() if isinstance(x, str) else x))
df.columns = df.columns.str.strip()

X = df[
    [
        'Your gender',
        'How important is exercise to you ?',
        'How healthy do you consider yourself?',
    ]
]
X = pd.get_dummies(X, drop_first=False)

age_col = 'Your age'
if age_col not in df.columns:
    age_col = [c for c in df.columns if c.strip().startswith('Your age')][0]

y = df[age_col].apply(lambda x: int(str(x).split()[0]))

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

with open('2.2.3_model.pkl', 'wb') as model_file:
    pickle.dump(rf_model, model_file)

y_pred = rf_model.predict(X_test)
pd.DataFrame(y_pred, columns=['预测结果']).to_csv(
    '2.2.3_results.txt', index=False
)

train_score = rf_model.score(X_train, y_train)
test_score = rf_model.score(X_test, y_test)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
with open('2.2.3_report.txt', 'w') as report_file:
    report_file.write(f'训练集得分: {train_score}\n')
    report_file.write(f'测试集得分: {test_score}\n')
    report_file.write(f'均方误差(MSE): {mse}\n')
    report_file.write(f'决定系数(R^2): {r2}\n')

xgb_model = xgb.XGBRegressor(n_estimators=100, random_state=42)
xgb_model.fit(X_train, y_train)
y_pred_xgb = xgb_model.predict(X_test)

pd.DataFrame(y_pred_xgb, columns=['预测结果']).to_csv(
    '2.2.3_results_xgb.txt', index=False
)

with open('2.2.3_report_xgb.txt', 'w') as xgb_report_file:
    xgb_report_file.write(
        f'XGBoost训练集得分: {xgb_model.score(X_train, y_train)}\n'
    )
    xgb_report_file.write(
        f'XGBoost测试集得分: {xgb_model.score(X_test, y_test)}\n'
    )
    xgb_report_file.write(
        f'XGBoost均方误差(MSE): '
        f'{mean_squared_error(y_test, y_pred_xgb)}\n'
    )
    xgb_report_file.write(
        f'XGBoost决定系数(R^2): '
        f'{r2_score(y_test, y_pred_xgb)}\n'
    )

## 学习提示

- 编程题：请在**本案例文件夹**下运行 Notebook，以便相对路径 `data/...` 正确。
- 主观题：示例答案仅供参考，可按考场要求改写；若未安装 `python-docx`，可先 `pip install python-docx`。
- 交卷前核对平台要求的截图、HTML、docx 命名与保存路径。